In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# 加载清洗后的数据
df = pd.read_csv('E_commerce_cleaned.csv')

print(f"原始数据形状: {df.shape}")
print(f"唯一用户数: {df['Customer ID'].nunique()}")
print(f"平均每个用户订单数: {df.shape[0] / df['Customer ID'].nunique():.2f}")

原始数据形状: (51289, 36)
唯一用户数: 795
平均每个用户订单数: 64.51


In [5]:
# 重新定义 create_customer_features 函数，修复列名不匹配的问题
def create_customer_features(df):
    """为每个用户创建聚合特征"""
    
    # 1. 用户基本信息（取最新记录）
    customer_info = df.sort_values('Order Date').groupby('Customer ID').agg({
        'Customer Name': 'last',
        'Segment': 'last',
        'Country': 'last',
        'Region': 'last',
        'City': 'last',
        'State': 'last',
        'Gender': 'last',
        'Age': 'last',
        'Age Group': 'last',
        'Education': 'last',
        'Marital Status': 'last'
    }).reset_index()
    
    # 2. 购买行为特征
    purchase_behavior = df.groupby('Customer ID').agg({
        'Order ID': 'count',  # 订单数量
        'Quantity': 'sum',  # 总购买数量
        'Sales': 'sum',  # 总销售额
        'Profit': 'sum',  # 总利润
        'Discount Amount': 'sum',  # 总折扣金额
        'Net Sales': 'sum',  # 净销售额
        'Shipping Cost': 'sum',  # 总运费
        'Discount': 'mean'  # 平均折扣率
    }).reset_index()
    
    purchase_behavior.columns = ['Customer ID', 'Total_Orders', 'Total_Quantity', 
                                'Total_Sales', 'Total_Profit', 'Total_Discount_Amount',
                                'Total_Net_Sales', 'Total_Shipping_Cost', 'Avg_Discount']
    
    # 3. 时间相关特征
    # 确保日期格式
    df['Order Date'] = pd.to_datetime(df['Order Date'])
    
    time_features = df.groupby('Customer ID').agg({
        'Order Date': ['min', 'max']  # 首次和最近购买时间
    }).reset_index()
    
    time_features.columns = ['Customer ID', 'First_Purchase_Date', 'Last_Purchase_Date']
    
    # 计算活跃天数
    time_features['Customer_Lifetime_Days'] = (
        time_features['Last_Purchase_Date'] - time_features['First_Purchase_Date']
    ).dt.days
    
    # 处理首次和最近购买相同的用户（只有一次购买）
    time_features.loc[time_features['Customer_Lifetime_Days'] == 0, 'Customer_Lifetime_Days'] = 1
    
    # 4. 购买频率特征
    purchase_frequency = df.groupby(['Customer ID', 'Order Date']).size().reset_index()
    purchase_frequency = purchase_frequency.groupby('Customer ID').agg({
        'Order Date': 'count'  # 购买天数
    }).reset_index()
    
    purchase_frequency.columns = ['Customer ID', 'Purchase_Days']
    
    # 计算平均购买间隔
    purchase_dates = df.groupby('Customer ID')['Order Date'].apply(list)
    
    def calculate_avg_purchase_interval(dates):
        if len(dates) <= 1:
            return 0
        dates.sort()
        intervals = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
        return np.mean(intervals) if intervals else 0
    
    avg_intervals = purchase_dates.apply(calculate_avg_purchase_interval)
    avg_intervals = avg_intervals.reset_index()
    avg_intervals.columns = ['Customer ID', 'Avg_Purchase_Interval_Days']
    
    # 5. 产品偏好特征
    product_preference = df.groupby(['Customer ID', 'Product']).agg({
        'Sales': 'sum',
        'Quantity': 'sum'
    }).reset_index()
    
    # 找到每个用户最常购买的产品
    favorite_product = product_preference.sort_values(['Customer ID', 'Quantity'], 
                                                     ascending=[True, False])
    favorite_product = favorite_product.drop_duplicates('Customer ID', keep='first')
    favorite_product = favorite_product[['Customer ID', 'Product']]
    favorite_product.columns = ['Customer ID', 'Favorite_Product']
    
    # 购买的产品种类数
    unique_products = df.groupby('Customer ID')['Product'].nunique().reset_index()
    unique_products.columns = ['Customer ID', 'Unique_Products_Purchased']
    
    # 6. 行为互动特征
    behavior_features = df.groupby('Customer ID').agg({
        'Browsing Time (min)': 'mean',  # 平均浏览时间
        'Like': 'sum',  # 总点赞数
        'Share': 'sum',  # 总分享数
        'Add to Cart': 'sum',  # 总加购数
        'Engagement Score': 'sum'  # 总互动分
    }).reset_index()
    
    behavior_features.columns = ['Customer ID', 'Avg_Browsing_Time', 'Total_Likes',
                                'Total_Shares', 'Total_Add_to_Cart', 'Total_Engagement_Score']
    
    # 7. 订单优先级特征 - 修复这里
    # 首先查看订单优先级的唯一值
    print(f"订单优先级唯一值: {df['Order Priority'].unique()}")
    
    order_priority = pd.crosstab(df['Customer ID'], df['Order Priority'])
    order_priority = order_priority.reset_index()
    
    # 动态创建列名
    priority_columns = ['Customer ID']
    for col in order_priority.columns[1:]:
        priority_columns.append(f'{col}_Orders')
    order_priority.columns = priority_columns
    
    print(f"订单优先级特征列: {priority_columns}")
    
    # 8. 订单时间特征（季度/月份偏好）- 修复这里
    monthly_orders = pd.crosstab(df['Customer ID'], df['Order Month'])
    monthly_orders = monthly_orders.reset_index()
    
    # 动态重命名列
    month_columns = ['Customer ID']
    for col in monthly_orders.columns[1:]:
        month_columns.append(f'Month_{col}_Orders')
    monthly_orders.columns = month_columns
    
    # 9. 计算关键指标
    # 合并所有特征
    customer_features = customer_info.merge(purchase_behavior, on='Customer ID')
    customer_features = customer_features.merge(time_features, on='Customer ID')
    customer_features = customer_features.merge(purchase_frequency, on='Customer ID')
    customer_features = customer_features.merge(avg_intervals, on='Customer ID')
    customer_features = customer_features.merge(favorite_product, on='Customer ID')
    customer_features = customer_features.merge(unique_products, on='Customer ID')
    customer_features = customer_features.merge(behavior_features, on='Customer ID')
    customer_features = customer_features.merge(order_priority, on='Customer ID')
    customer_features = customer_features.merge(monthly_orders, on='Customer ID', how='left')
    
    # 10. 计算衍生特征
    # 计算平均订单价值
    customer_features['Avg_Order_Value'] = (
        customer_features['Total_Sales'] / customer_features['Total_Orders']
    ).round(2)
    
    # 计算平均每单利润
    customer_features['Avg_Order_Profit'] = (
        customer_features['Total_Profit'] / customer_features['Total_Orders']
    ).round(2)
    
    # 计算平均每单数量
    customer_features['Avg_Order_Quantity'] = (
        customer_features['Total_Quantity'] / customer_features['Total_Orders']
    ).round(2)
    
    # 计算利润率
    customer_features['Profit_Margin'] = (
        customer_features['Total_Profit'] / customer_features['Total_Sales']
    ).round(4)
    
    # 计算运费占比
    customer_features['Shipping_Cost_Ratio'] = (
        customer_features['Total_Shipping_Cost'] / customer_features['Total_Sales']
    ).round(4)
    
    # 计算折扣率
    customer_features['Discount_Ratio'] = (
        customer_features['Total_Discount_Amount'] / customer_features['Total_Sales']
    ).round(4)
    
    # 计算购买频率（每天购买次数）
    customer_features['Purchase_Frequency'] = (
        customer_features['Total_Orders'] / customer_features['Customer_Lifetime_Days']
    ).round(4)
    
    # 计算每单浏览时间
    customer_features['Browsing_Time_per_Order'] = (
        customer_features['Avg_Browsing_Time']
    ).round(2)
    
    # 11. 创建客户价值分组
    # 基于RFM模型创建特征
    # Recency: 距离最近一次购买的天数（相对于数据集最新日期）
    latest_date = df['Order Date'].max()
    customer_features['Days_Since_Last_Purchase'] = (
        latest_date - customer_features['Last_Purchase_Date']
    ).dt.days
    
    # Monetary: 总消费金额
    customer_features['Monetary_Value'] = customer_features['Total_Sales']
    
    # Frequency: 购买频率
    customer_features['Purchase_Frequency_RFM'] = customer_features['Total_Orders']
    
    return customer_features

# 创建用户特征
print("开始创建用户特征...")
customer_features = create_customer_features(df)

print(f"\n用户特征数据集形状: {customer_features.shape}")
print(f"\n前几列特征:")
for i, col in enumerate(customer_features.columns[:20], 1):
    print(f"{i:3}. {col}")

print(f"\n... 还有 {len(customer_features.columns) - 20} 列 ...")

# 检查数据质量
print("\n数据质量检查:")
print(f"缺失值总数: {customer_features.isnull().sum().sum()}")
print(f"重复用户数: {customer_features['Customer ID'].duplicated().sum()}")

# 查看基本统计
print("\n数值特征统计:")
numeric_cols = customer_features.select_dtypes(include=[np.number]).columns.tolist()
print(customer_features[numeric_cols[:10]].describe())

开始创建用户特征...
订单优先级唯一值: ['Medium' 'Critical' 'High' 'Low']
订单优先级特征列: ['Customer ID', 'Critical_Orders', 'High_Orders', 'Low_Orders', 'Medium_Orders']

用户特征数据集形状: (795, 59)

前几列特征:
  1. Customer ID
  2. Customer Name
  3. Segment
  4. Country
  5. Region
  6. City
  7. State
  8. Gender
  9. Age
 10. Age Group
 11. Education
 12. Marital Status
 13. Total_Orders
 14. Total_Quantity
 15. Total_Sales
 16. Total_Profit
 17. Total_Discount_Amount
 18. Total_Net_Sales
 19. Total_Shipping_Cost
 20. Avg_Discount

... 还有 39 列 ...

数据质量检查:
缺失值总数: 1
重复用户数: 0

数值特征统计:
              Age  Total_Orders  Total_Quantity   Total_Sales  Total_Profit  \
count  795.000000    795.000000      795.000000    795.000000    795.000000   
mean    31.343396     64.514465      193.371069  10091.947170   4691.897987   
std     11.835738     13.434166       42.117317   2202.601987   1079.222745   
min     18.000000     29.000000       80.000000   4249.000000   1825.600000   
25%     22.000000     55.000000      163.000

In [6]:
# 创建用户特征
customer_features = create_customer_features(df)

print(f"用户特征数据集形状: {customer_features.shape}")
print(f"\n特征列列表:")
for i, col in enumerate(customer_features.columns, 1):
    print(f"{i:3}. {col}")

# 检查数据质量
print("\n数据质量检查:")
print(f"缺失值总数: {customer_features.isnull().sum().sum()}")
print(f"重复用户数: {customer_features['Customer ID'].duplicated().sum()}")

# 查看基本统计
print("\n数值特征统计:")
numeric_cols = customer_features.select_dtypes(include=[np.number]).columns.tolist()
print(customer_features[numeric_cols[:10]].describe())

订单优先级唯一值: ['Medium' 'Critical' 'High' 'Low']
订单优先级特征列: ['Customer ID', 'Critical_Orders', 'High_Orders', 'Low_Orders', 'Medium_Orders']
用户特征数据集形状: (795, 59)

特征列列表:
  1. Customer ID
  2. Customer Name
  3. Segment
  4. Country
  5. Region
  6. City
  7. State
  8. Gender
  9. Age
 10. Age Group
 11. Education
 12. Marital Status
 13. Total_Orders
 14. Total_Quantity
 15. Total_Sales
 16. Total_Profit
 17. Total_Discount_Amount
 18. Total_Net_Sales
 19. Total_Shipping_Cost
 20. Avg_Discount
 21. First_Purchase_Date
 22. Last_Purchase_Date
 23. Customer_Lifetime_Days
 24. Purchase_Days
 25. Avg_Purchase_Interval_Days
 26. Favorite_Product
 27. Unique_Products_Purchased
 28. Avg_Browsing_Time
 29. Total_Likes
 30. Total_Shares
 31. Total_Add_to_Cart
 32. Total_Engagement_Score
 33. Critical_Orders
 34. High_Orders
 35. Low_Orders
 36. Medium_Orders
 37. Month_1_Orders
 38. Month_2_Orders
 39. Month_3_Orders
 40. Month_4_Orders
 41. Month_5_Orders
 42. Month_6_Orders
 43. Month_7_Orders
 4

In [7]:
def create_customer_segments(customer_features):
    """基于用户特征创建客户细分"""
    
    # 创建副本，避免修改原始数据
    segments = customer_features.copy()
    
    # 1. 基于RFM的客户细分
    # 计算RFM百分位数
    segments['R_Score'] = pd.qcut(segments['Days_Since_Last_Purchase'], 
                                   q=5, labels=[5, 4, 3, 2, 1])
    segments['F_Score'] = pd.qcut(segments['Purchase_Frequency_RFM'], 
                                   q=5, labels=[1, 2, 3, 4, 5])
    segments['M_Score'] = pd.qcut(segments['Monetary_Value'], 
                                   q=5, labels=[1, 2, 3, 4, 5])
    
    # 转换分数为数值
    segments['R_Score'] = segments['R_Score'].astype(int)
    segments['F_Score'] = segments['F_Score'].astype(int)
    segments['M_Score'] = segments['M_Score'].astype(int)
    
    # 计算RFM总分
    segments['RFM_Score'] = segments['R_Score'] + segments['F_Score'] + segments['M_Score']
    
    # RFM客户细分
    def rfm_segment(row):
        if row['RFM_Score'] >= 12:
            return 'Champions'
        elif row['RFM_Score'] >= 9:
            return 'Loyal Customers'
        elif row['RFM_Score'] >= 6:
            return 'Potential Loyalists'
        elif row['RFM_Score'] >= 4:
            return 'At Risk Customers'
        else:
            return 'Lost Customers'
    
    segments['RFM_Segment'] = segments.apply(rfm_segment, axis=1)
    
    # 2. 基于价值贡献的细分
    # 计算总销售额百分位
    sales_percentile = segments['Total_Sales'].rank(pct=True)
    
    def value_segment(percentile):
        if percentile >= 0.8:
            return 'VIP'
        elif percentile >= 0.5:
            return 'High Value'
        elif percentile >= 0.2:
            return 'Medium Value'
        else:
            return 'Low Value'
    
    segments['Value_Segment'] = sales_percentile.apply(value_segment)
    
    # 3. 基于购买行为的细分
    def behavior_segment(row):
        if row['Total_Orders'] >= 5 and row['Avg_Purchase_Interval_Days'] <= 30:
            return 'Frequent Buyer'
        elif row['Avg_Order_Value'] >= segments['Avg_Order_Value'].median():
            return 'High Spender'
        elif row['Purchase_Frequency'] >= segments['Purchase_Frequency'].median():
            return 'Regular Buyer'
        else:
            return 'Occasional Buyer'
    
    segments['Behavior_Segment'] = segments.apply(behavior_segment, axis=1)
    
    # 4. 基于利润率的细分
    profit_margin_percentile = segments['Profit_Margin'].rank(pct=True)
    
    def profitability_segment(percentile):
        if percentile >= 0.8:
            return 'Highly Profitable'
        elif percentile >= 0.5:
            return 'Profitable'
        elif percentile >= 0.2:
            return 'Break-even'
        else:
            return 'Low Margin'
    
    segments['Profitability_Segment'] = profit_margin_percentile.apply(profitability_segment)
    
    # 5. 基于产品偏好的细分
    segments['Product_Variety_Level'] = pd.qcut(segments['Unique_Products_Purchased'], 
                                                 q=3, labels=['Low Variety', 'Medium Variety', 'High Variety'])
    
    return segments

# 创建客户细分
customer_segments = create_customer_segments(customer_features)

print(f"客户细分数据集形状: {customer_segments.shape}")
print(f"\n新增的细分特征:")
new_segment_cols = ['R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'RFM_Segment',
                    'Value_Segment', 'Behavior_Segment', 'Profitability_Segment',
                    'Product_Variety_Level']
print(customer_segments[new_segment_cols].head())

客户细分数据集形状: (795, 68)

新增的细分特征:
   R_Score  F_Score  M_Score  RFM_Score          RFM_Segment Value_Segment  \
0        3        2        2          7  Potential Loyalists  Medium Value   
1        1        1        1          3       Lost Customers     Low Value   
2        4        1        1          6  Potential Loyalists     Low Value   
3        3        1        1          5    At Risk Customers     Low Value   
4        5        3        3         11      Loyal Customers  Medium Value   

  Behavior_Segment Profitability_Segment Product_Variety_Level  
0   Frequent Buyer            Break-even           Low Variety  
1   Frequent Buyer     Highly Profitable           Low Variety  
2   Frequent Buyer            Low Margin           Low Variety  
3   Frequent Buyer            Break-even        Medium Variety  
4   Frequent Buyer            Break-even        Medium Variety  


In [8]:
def create_prediction_features(customer_segments):
    """创建用于预测的特征"""
    
    prediction_features = customer_segments.copy()
    
    # 1. 响应变量：是否会在未来30天购买（基于最后购买时间）
    # 由于我们只有历史数据，这里创建一个模拟的未来购买标签
    # 基于用户的购买模式来预测
    
    # 创建购买活跃度指标
    prediction_features['Purchase_Recency_Ratio'] = (
        prediction_features['Days_Since_Last_Purchase'] / 
        prediction_features['Customer_Lifetime_Days']
    ).fillna(0)
    
    # 2. 创建交叉特征
    # 价值×频率交叉
    prediction_features['Value_Frequency_Score'] = (
        prediction_features['Avg_Order_Value'] * 
        prediction_features['Purchase_Frequency']
    )
    
    # 利润×忠诚度交叉
    prediction_features['Profit_Loyalty_Score'] = (
        prediction_features['Profit_Margin'] * 
        prediction_features['Customer_Lifetime_Days']
    )
    
    # 3. 标准化数值特征（用于聚类）
    from sklearn.preprocessing import StandardScaler
    
    # 选择要标准化的特征
    features_to_scale = ['Total_Sales', 'Total_Profit', 'Total_Orders',
                        'Avg_Order_Value', 'Purchase_Frequency', 'Avg_Browsing_Time']
    
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(prediction_features[features_to_scale])
    
    # 添加到数据集
    for i, col in enumerate(features_to_scale):
        prediction_features[f'{col}_Scaled'] = scaled_features[:, i]
    
    # 4. 创建聚类特征
    from sklearn.cluster import KMeans
    
    # 使用标准化后的特征进行聚类
    clustering_features = prediction_features[[f'{col}_Scaled' for col in features_to_scale]]
    
    # 进行K-means聚类（假设4个群组）
    kmeans = KMeans(n_clusters=4, random_state=42)
    prediction_features['Cluster_Group'] = kmeans.fit_predict(clustering_features)
    
    # 5. 创建客户生命周期价值预测特征
    # CLV = 平均订单价值 × 购买频率 × 客户生命周期
    prediction_features['Predicted_CLV'] = (
        prediction_features['Avg_Order_Value'] *
        prediction_features['Purchase_Frequency'] *
        prediction_features['Customer_Lifetime_Days']
    )
    
    # 6. 创建流失风险特征
    def churn_risk_score(row):
        score = 0
        # 最近没有购买
        if row['Days_Since_Last_Purchase'] > 90:
            score += 3
        elif row['Days_Since_Last_Purchase'] > 60:
            score += 2
        elif row['Days_Since_Last_Purchase'] > 30:
            score += 1
        
        # 购买频率下降
        if row['Purchase_Frequency'] < prediction_features['Purchase_Frequency'].median():
            score += 2
        
        # 平均订单价值下降
        if row['Avg_Order_Value'] < prediction_features['Avg_Order_Value'].median():
            score += 1
        
        # 互动减少
        if row['Total_Engagement_Score'] < prediction_features['Total_Engagement_Score'].median():
            score += 1
        
        return min(score, 5)  # 限制在0-5分
    
    prediction_features['Churn_Risk_Score'] = prediction_features.apply(churn_risk_score, axis=1)
    
    # 7. 创建升级潜力特征
    def upgrade_potential(row):
        score = 0
        
        # 高价值但低频
        if (row['Avg_Order_Value'] > prediction_features['Avg_Order_Value'].quantile(0.75) and
            row['Purchase_Frequency'] < prediction_features['Purchase_Frequency'].median()):
            score += 3
        
        # 高频但低价值
        elif (row['Purchase_Frequency'] > prediction_features['Purchase_Frequency'].quantile(0.75) and
              row['Avg_Order_Value'] < prediction_features['Avg_Order_Value'].median()):
            score += 2
        
        # 最近购买且互动高
        elif (row['Days_Since_Last_Purchase'] < 30 and
              row['Total_Engagement_Score'] > prediction_features['Total_Engagement_Score'].median()):
            score += 2
        
        return min(score, 3)  # 限制在0-3分
    
    prediction_features['Upgrade_Potential_Score'] = prediction_features.apply(upgrade_potential, axis=1)
    
    return prediction_features

# 创建预测特征
customer_prediction = create_prediction_features(customer_segments)

print(f"预测特征数据集形状: {customer_prediction.shape}")
print(f"\n新增的预测特征:")
new_prediction_cols = ['Value_Frequency_Score', 'Profit_Loyalty_Score', 'Cluster_Group',
                      'Predicted_CLV', 'Churn_Risk_Score', 'Upgrade_Potential_Score']
print(customer_prediction[new_prediction_cols].head())

预测特征数据集形状: (795, 81)

新增的预测特征:
   Value_Frequency_Score  Profit_Loyalty_Score  Cluster_Group  Predicted_CLV  \
0               8.388264              492.5616              0    8958.665952   
1               7.526304              511.1010              1    7842.408768   
2               7.240980              460.2087              1    7755.089580   
3               7.542234              496.6227              1    8077.732614   
4               8.785070              499.7463              0    9496.660670   

   Churn_Risk_Score  Upgrade_Potential_Score  
0                 3                        2  
1                 4                        3  
2                 2                        2  
3                 3                        2  
4                 3                        2  


In [9]:
# 保存完整用户特征数据集
customer_features.to_csv('customer_features_complete.csv', index=False)
customer_segments.to_csv('customer_features_with_segments.csv', index=False)
customer_prediction.to_csv('customer_features_for_prediction.csv', index=False)

print("\n用户特征数据集已保存:")
print("1. customer_features_complete.csv - 基础聚合特征")
print("2. customer_features_with_segments.csv - 包含客户细分")
print("3. customer_features_for_prediction.csv - 包含预测特征")

# 创建汇总报告
def create_feature_summary(df, name):
    """创建特征汇总报告"""
    summary = {
        '数据集名称': name,
        '总行数': df.shape[0],
        '总列数': df.shape[1],
        '数值特征数': df.select_dtypes(include=[np.number]).shape[1],
        '分类特征数': df.select_dtypes(include=['object', 'category']).shape[1],
        '日期特征数': df.select_dtypes(include=['datetime64']).shape[1],
        '缺失值总数': df.isnull().sum().sum(),
        '重复行数': df.duplicated().sum()
    }
    return summary

# 生成所有数据集的汇总
datasets = [
    (customer_features, '基础聚合特征'),
    (customer_segments, '客户细分特征'),
    (customer_prediction, '预测特征')
]

print("\n数据集汇总报告:")
print("-" * 50)
for df, name in datasets:
    summary = create_feature_summary(df, name)
    for key, value in summary.items():
        print(f"{key}: {value}")
    print("-" * 50)

# 查看最重要的特征示例
print("\n最重要的用户特征示例（前10个用户）:")
important_features = ['Customer ID', 'Customer Name', 'Total_Orders', 'Total_Sales', 
                     'Total_Profit', 'Avg_Order_Value', 'Customer_Lifetime_Days',
                     'Purchase_Frequency', 'RFM_Segment', 'Value_Segment', 'Churn_Risk_Score']
print(customer_prediction[important_features].head(10).to_string())


用户特征数据集已保存:
1. customer_features_complete.csv - 基础聚合特征
2. customer_features_with_segments.csv - 包含客户细分
3. customer_features_for_prediction.csv - 包含预测特征

数据集汇总报告:
--------------------------------------------------
数据集名称: 基础聚合特征
总行数: 795
总列数: 59
数值特征数: 45
分类特征数: 12
日期特征数: 2
缺失值总数: 1
重复行数: 0
--------------------------------------------------
数据集名称: 客户细分特征
总行数: 795
总列数: 68
数值特征数: 49
分类特征数: 17
日期特征数: 2
缺失值总数: 1
重复行数: 0
--------------------------------------------------
数据集名称: 预测特征
总行数: 795
总列数: 81
数值特征数: 62
分类特征数: 17
日期特征数: 2
缺失值总数: 1
重复行数: 0
--------------------------------------------------

最重要的用户特征示例（前10个用户）:
  Customer ID       Customer Name  Total_Orders  Total_Sales  Total_Profit  Avg_Order_Value  Customer_Lifetime_Days  Purchase_Frequency          RFM_Segment Value_Segment  Churn_Risk_Score
0    AB-00363     Mcintyre Yedwab            58       8960.0        4132.7           154.48                    1068              0.0543  Potential Loyalists  Medium Value                 3
1    